In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约1-2分钟）
!pip install -q numpy scipy matplotlib soundfile torch torchaudio
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

# Python基础——从计算器到程序

## 学习目标

- 理解Python从"计算器式脚本"到"程序化思维"的转变
- 掌握变量、数据类型、运算的基本用法
- 用NumPy生成和操作数组，理解向量化思维
- 掌握控制流（if/for/while）的基本用法
- **重点**：理解为什么要写函数，从"复制粘贴"到"函数调用"
- 掌握列表和字典的基本操作

## 本节约定

所有练习都使用**音频信号**作为素材：
- 不用斐波那契数列学循环——用"生成不同频率的正弦波"学循环
- 不用学生成绩单学字典——用"音频文件的元数据"学字典
- 不用字符串反转学函数——用"信号生成与处理"学函数


## 1. Python环境确认

首先确认你的Python环境和必要的库已经安装好。

In [ ]:
import sys
print(f'Python版本: {sys.version}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
print('所有库导入成功！')

## 2. 变量、数据类型与运算

### 2.1 变量

在音频处理中，我们用变量来存储音频的基本参数。

In [ ]:
# 音频基本参数
sample_rate = 16000       # 采样率 (Hz)
duration = 1.0            # 时长 (秒)
frequency = 440.0         # 频率 (Hz) — A4音
amplitude = 0.5           # 振幅

print(f'采样率: {sample_rate} Hz')
print(f'时长: {duration} 秒')
print(f'频率: {frequency} Hz (A4音)')
print(f'振幅: {amplitude}')

### 2.2 数据类型

Python的基本数据类型：
- `int`：整数（如采样率16000）
- `float`：浮点数（如频率440.0）
- `str`：字符串（如文件名"speech.wav"）
- `bool`：布尔值（如 is_speech = True）

In [ ]:
# 数据类型示例
sample_rate = 16000        # int
frequency = 440.0          # float
filename = 'speech.wav'    # str
is_speech = True           # bool

print(f'type(sample_rate) = {type(sample_rate)}')
print(f'type(frequency) = {type(frequency)}')
print(f'type(filename) = {type(filename)}')
print(f'type(is_speech) = {type(is_speech)}')

### 2.3 运算

音频处理中的常见运算：

In [ ]:
# 计算音频信号的总样本数
num_samples = int(sample_rate * duration)
print(f'总样本数: {num_samples}')

# 计算每个周期的样本数
samples_per_cycle = sample_rate / frequency
print(f'每个周期的样本数: {samples_per_cycle:.2f}')

# 分贝转换
import math
db = 20 * math.log10(amplitude)
print(f'振幅 {amplitude} 对应 {db:.1f} dB')

## 3. NumPy数组：从标量到向量

**关键概念**：音频信号就是一组数字（数组），NumPy让我们可以像写数学公式一样操作整个数组，而不是逐个元素循环。

这就是"向量化思维"——物理背景的学生应该很容易理解，这就像向量运算和标量运算的区别。

In [ ]:
# 生成时间轴
t = np.linspace(0, duration, num_samples, endpoint=False)
print(f'时间轴长度: {len(t)}')
print(f'前5个时间点: {t[:5]}')

In [ ]:
# 生成正弦波：纯音（A4 = 440 Hz）
sine_wave = amplitude * np.sin(2 * np.pi * frequency * t)
print(f'波形长度: {len(sine_wave)}')
print(f'前5个样本值: {sine_wave[:5]}')

In [ ]:
# 可视化
plt.figure(figsize=(12, 4))
plt.plot(t[:500], sine_wave[:500])  # 只画前500个样本
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title(f'Sine Wave: {frequency} Hz, Amplitude: {amplitude}')
plt.grid(True)
plt.show()

### 练习1：向量化运算

NumPy的强大之处在于可以对整个数组做运算，不需要循环。

**任务**：生成两个不同频率的正弦波，将它们相加，得到一个复合波。

In [ ]:
# ===== 练习1：向量化运算 =====
# 提示：
# 1. 生成一个 261.63 Hz (C4音) 的正弦波
# 2. 生成一个 329.63 Hz (E4音) 的正弦波
# 3. 将两个波相加
# 4. 画出结果

freq_C4 = 261.63   # C4音
freq_E4 = 329.63   # E4音

# TODO: 生成C4和E4的正弦波
wave_C4 = amplitude * np.sin(2 * np.pi * freq_C4 * t)
wave_E4 = amplitude * np.sin(2 * np.pi * freq_E4 * t)

# TODO: 将两个波相加
composite = wave_C4 + wave_E4

# 可视化
plt.figure(figsize=(12, 4))
plt.plot(t[:500], composite[:500])
plt.title('Composite Wave (C4 + E4)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

## 4. 控制流：if / for / while

### 4.1 if 条件判断

在音频处理中，我们经常需要根据条件做不同的处理。

In [ ]:
# 判断频率范围
def classify_frequency(freq):
    if freq < 300:
        return '低频 (Bass)'
    elif freq < 1000:
        return '中频 (Mid)'
    elif freq < 4000:
        return '中高频 (Upper Mid)'
    else:
        return '高频 (Treble)'

# 测试
test_freqs = [100, 440, 2000, 8000]
for f in test_freqs:
    print(f'{f} Hz → {classify_frequency(f)}')

### 4.2 for 循环

在音频处理中，for循环最常见的用途是批量处理。

In [ ]:
# 批量生成不同频率的正弦波
frequencies = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
note_names = ['C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5']

waves = {}  # 用字典存储不同频率的波
for freq, name in zip(frequencies, note_names):
    wave = amplitude * np.sin(2 * np.pi * freq * t)
    waves[name] = wave
    print(f'{name} ({freq} Hz): 长度={len(wave)}, 最大值={np.max(np.abs(wave)):.3f}')

### 4.3 while 循环

while循环在音频处理中相对少见，但有些场景（如找到第一个过零点）会用到。

In [ ]:
# 找到波形中第一个过零点的位置
i = 0
while sine_wave[i] > 0 and i < len(sine_wave) - 1:
    i += 1

zero_crossing_time = t[i]
print(f'第一个过零点出现在 t = {zero_crossing_time:.6f} 秒')
print(f'对应样本索引: {i}')
print(f'验证: sine_wave[{i-1}] = {sine_wave[i-1]:.6f}, sine_wave[{i}] = {sine_wave[i]:.6f}')

## 5. 函数——从"复制粘贴"到"函数调用"

**这是本节课最重要的部分。**

物理背景的学生最常见的编程坏习惯就是把所有代码写在脚本的最外层，从头到尾执行一遍——没有函数、没有模块化。

我们来看一个具体的例子。

### 5.1 坏习惯：复制粘贴

假设你需要生成3个不同频率的正弦波并画图。最直观的做法是：

In [ ]:
# ===== 坏习惯：复制粘贴 =====
# 生成 440 Hz 的正弦波并画图
wave_440 = 0.5 * np.sin(2 * np.pi * 440 * t)
plt.figure(figsize=(10, 3))
plt.plot(t[:500], wave_440[:500])
plt.title('440 Hz Sine Wave')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

# 生成 261 Hz 的正弦波并画图（复制上面的代码，改数字）
wave_261 = 0.5 * np.sin(2 * np.pi * 261 * t)
plt.figure(figsize=(10, 3))
plt.plot(t[:500], wave_261[:500])
plt.title('261 Hz Sine Wave')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

# 生成 880 Hz 的正弦波并画图（再复制一次...）
wave_880 = 0.5 * np.sin(2 * np.pi * 880 * t)
plt.figure(figsize=(10, 3))
plt.plot(t[:500], wave_880[:500])
plt.title('880 Hz Sine Wave')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

### 5.2 好习惯：封装成函数

现在把上面的重复代码封装成一个函数：

In [ ]:
def plot_sine_wave(freq, amp=0.5, sr=16000, dur=1.0, num_samples=500):
    """生成正弦波并画图
    
    参数:
        freq: 频率 (Hz)
        amp: 振幅 (默认 0.5)
        sr: 采样率 (默认 16000 Hz)
        dur: 时长 (默认 1.0 秒)
        num_samples: 画图时显示的样本数
    """
    t = np.linspace(0, dur, int(sr * dur), endpoint=False)
    wave = amp * np.sin(2 * np.pi * freq * t)
    
    plt.figure(figsize=(10, 3))
    plt.plot(t[:num_samples], wave[:num_samples])
    plt.title(f'{freq} Hz Sine Wave')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.show()
    
    return wave

# 现在只需要一行代码
wave_440 = plot_sine_wave(440)
wave_261 = plot_sine_wave(261)
wave_880 = plot_sine_wave(880)

### 对比

| 方式 | 代码行数 | 如果要改画图参数？ |
|------|---------|-------------------|
| 复制粘贴 | ~24行 | 需要改3个地方 |
| 函数封装 | ~3行 | 只需改函数定义1处 |

**函数不只是"把代码打包"，而是"把思路分块"**——每个函数做一件事，函数名就是这件事的描述。

### 5.3 函数的参数和返回值

函数可以有多个参数和返回值。

In [ ]:
def generate_sine_wave(freq, amp=0.5, sr=16000, dur=1.0):
    """生成正弦波
    
    参数:
        freq: 频率 (Hz)
        amp: 振幅
        sr: 采样率
        dur: 时长
    
    返回:
        t: 时间轴
        wave: 波形数据
    """
    t = np.linspace(0, dur, int(sr * dur), endpoint=False)
    wave = amp * np.sin(2 * np.pi * freq * t)
    return t, wave

# 使用
t, wave = generate_sine_wave(440)
print(f'波形长度: {len(wave)}')
print(f'前5个样本: {wave[:5]}')

### 练习2：写一个函数

**任务**：写一个函数 `generate_composite_wave(freqs, amps, sr=16000, dur=1.0)`，它：
1. 接收频率列表 `freqs` 和对应的振幅列表 `amps`
2. 生成每个频率对应的正弦波
3. 将所有波相加
4. 返回时间轴 `t` 和复合波 `composite`

**提示**：参考上面 `generate_sine_wave` 函数的写法，用 for 循环遍历 freqs 和 amps。

In [ ]:
# ===== 练习2：写一个函数 =====
def generate_composite_wave(freqs, amps, sr=16000, dur=1.0):
    """生成复合波（多个正弦波叠加）
    
    参数:
        freqs: 频率列表
        amps: 对应的振幅列表
        sr: 采样率
        dur: 时长
    
    返回:
        t: 时间轴
        composite: 复合波
    """
    # TODO: 生成时间轴
    t = np.linspace(0, dur, int(sr * dur), endpoint=False)
    
    # TODO: 生成复合波
    composite = np.zeros_like(t)
    for freq, amp in zip(freqs, amps):
        # TODO: 生成每个频率的正弦波并叠加
        wave = amp * np.sin(2 * np.pi * freq * t)
        composite += wave
    
    return t, composite

# 测试
freqs = [261.63, 329.63, 392.00]  # C大三和弦: C4, E4, G4
amps = [0.5, 0.5, 0.5]
t, chord = generate_composite_wave(freqs, amps)

plt.figure(figsize=(12, 4))
plt.plot(t[:500], chord[:500])
plt.title('C Major Chord (C4 + E4 + G4)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

## 6. 列表与字典

在音频处理中，我们经常需要存储和管理音频文件的信息。

### 6.1 列表

列表用于存储有序的数据集合。

In [ ]:
# 常见的音频采样率
common_sample_rates = [8000, 16000, 22050, 44100, 48000, 96000]
print(f'常见采样率: {common_sample_rates}')
print(f'CD音质采样率: {common_sample_rates[3]} Hz')
print(f'电话音质采样率: {common_sample_rates[0]} Hz')

### 6.2 字典

字典用于存储键值对，非常适合存储结构化的数据。

In [ ]:
# 用字典存储wav文件的信息
audio_info = {
    'filename': 'speech.wav',
    'sample_rate': 16000,
    'duration': 3.5,
    'num_channels': 1,
    'num_samples': None,  # 稍后计算
    'is_speech': True,
}

# 计算并更新
audio_info['num_samples'] = audio_info['sample_rate'] * audio_info['duration']

print('音频文件信息:')
for key, value in audio_info.items():
    print(f'  {key}: {value}')

### 练习3：用字典管理多个音频文件

**任务**：创建一个字典 `audio_files`，其中每个键是文件名，值是另一个字典（包含 sample_rate, duration, is_speech）。然后用 for 循环遍历，打印每个文件的总样本数。

In [ ]:
# ===== 练习3：用字典管理多个音频文件 =====
audio_files = {
    'speech_clean.wav': {
        'sample_rate': 16000,
        'duration': 3.5,
        'is_speech': True,
    },
    'speech_noisy.wav': {
        'sample_rate': 16000,
        'duration': 5.0,
        'is_speech': True,
    },
    'noise_white.wav': {
        'sample_rate': 44100,
        'duration': 10.0,
        'is_speech': False,
    },
}

# TODO: 用 for 循环遍历，打印每个文件的总样本数
for filename, info in audio_files.items():
    num_samples = int(info['sample_rate'] * info['duration'])
    print(f"{filename}: {num_samples} 样本, 语音={info['is_speech']}")

## 7. 综合练习

**任务**：写一个函数 `generate_sine_wave(freq, duration, sample_rate, amplitude)`，它：
1. 生成指定频率和时长的正弦波
2. 返回时间轴 `t` 和波形数据 `wave`
3. 画出波形的前500个样本

**要求**：不参考上面的代码，独立完成。

In [ ]:
# ===== 综合练习 =====
def generate_sine_wave(freq, duration, sample_rate, amplitude):
    """生成正弦波
    
    参数:
        freq: 频率 (Hz)
        duration: 时长 (秒)
        sample_rate: 采样率 (Hz)
        amplitude: 振幅
    
    返回:
        t: 时间轴
        wave: 波形数据
    """
    # TODO: 生成时间轴
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    
    # TODO: 生成正弦波
    wave = amplitude * np.sin(2 * np.pi * freq * t)
    
    # TODO: 画图
    plt.figure(figsize=(12, 4))
    plt.plot(t[:500], wave[:500])
    plt.title(f'{freq} Hz Sine Wave, Amplitude: {amplitude}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.show()
    
    return t, wave

# 测试你的函数
t, wave = generate_sine_wave(440, 0.5, 16000, 0.5)

## 本节要点

1. **变量**存储数据，选择有意义的变量名
2. **NumPy数组**是音频处理的核心数据结构，学会向量化思维
3. **控制流**（if/for/while）用于条件判断和批量处理
4. **函数**是最重要的概念——从"复制粘贴"到"函数调用"，函数把思路分块
5. **列表和字典**用于组织和管理数据


---
下一节：[02-oop-audio.ipynb](02-oop-audio.ipynb) — 面向对象与代码组织